# scikit-learn notebook

In [2]:
import numpy as np
import pandas as pd
 
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GridSearchCV)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
 
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
 
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, mean_squared_error,
                             mean_absolute_error, r2_score)

In [3]:
RANDOM_STATE = 42   # set everywhere for reproducible, grade-able results

# Part A - Classification

# 1. Load. as_frame=True gives a pandas DataFrame (nicer to work with).


In [4]:
data = load_breast_cancer(as_frame=True)
X = data.data          # 30 numeric features
y = data.target        # 0 = malignant, 1 = benign
print("X shape ->", X.shape, "| classes ->", dict(y.value_counts()))
print("first few features ->", list(X.columns[:4]))

X shape -> (569, 30) | classes -> {1: np.int64(357), 0: np.int64(212)}
first few features -> ['mean radius', 'mean texture', 'mean perimeter', 'mean area']


# 2. Train/test split. stratify=y keeps the class ratio equal in both splits

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)
print("\ntrain rows ->", len(X_train), "| test rows ->", len(X_test))


train rows -> 426 | test rows -> 143


 
# 3. Scale features. Fit the scaler on TRAIN ONLY, then transform both.

In [ ]:
#    Fitting on the full data would leak test information -> optimistic scores.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit + transform train
X_test_s = scaler.transform(X_test)         # transform test with train's stats

# 4 + 5. Fit two models and predict.

In [7]:
# --- Baseline: Logistic Regression ---
logreg = LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
logreg.fit(X_train_s, y_train)
pred_lr = logreg.predict(X_test_s)
proba_lr = logreg.predict_proba(X_test_s)[:, 1]   # prob of class 1 -> for ROC-AUC
 
# --- Stronger: Random Forest (trees don't need scaling, but it's harmless) ---
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)                 # fit on unscaled is fine for trees
pred_rf = rf.predict(X_test)
proba_rf = rf.predict_proba(X_test)[:, 1]

# 6. Evaluate. For classification: accuracy, precision, recall, F1, ROC-AUC.

In [8]:
 
def classification_scores(name, y_true, pred, proba):
    print(f"\n{name}")
    print(f"  accuracy  : {accuracy_score(y_true, pred):.3f}")
    print(f"  precision : {precision_score(y_true, pred):.3f}")
    print(f"  recall    : {recall_score(y_true, pred):.3f}")
    print(f"  f1        : {f1_score(y_true, pred):.3f}")
    print(f"  roc_auc   : {roc_auc_score(y_true, proba):.3f}  (uses probabilities!)")
 
classification_scores("Logistic Regression", y_test, pred_lr, proba_lr)
classification_scores("Random Forest", y_test, pred_rf, proba_rf)
 
# Confusion matrix: rows = actual, cols = predicted. [[TN, FP], [FN, TP]]
print("\nConfusion matrix (Random Forest):\n", confusion_matrix(y_test, pred_rf))
 
# classification_report bundles precision/recall/f1 per class in one string.
print("\nclassification_report (Random Forest):\n",
      classification_report(y_test, pred_rf, target_names=data.target_names))
 

 


Logistic Regression
  accuracy  : 0.986
  precision : 0.989
  recall    : 0.989
  f1        : 0.989
  roc_auc   : 0.998  (uses probabilities!)

Random Forest
  accuracy  : 0.958
  precision : 0.957
  recall    : 0.978
  f1        : 0.967
  roc_auc   : 0.995  (uses probabilities!)

Confusion matrix (Random Forest):
 [[49  4]
 [ 2 88]]

classification_report (Random Forest):
               precision    recall  f1-score   support

   malignant       0.96      0.92      0.94        53
      benign       0.96      0.98      0.97        90

    accuracy                           0.96       143
   macro avg       0.96      0.95      0.95       143
weighted avg       0.96      0.96      0.96       143



# 7. Feature importance — which features drove the Random Forest?

In [9]:

importances = pd.Series(rf.feature_importances_, index=X.columns)
print("Top 5 features by importance:\n",
      importances.sort_values(ascending=False).head(5).round(3))

Top 5 features by importance:
 worst perimeter         0.146
worst area              0.137
worst concave points    0.107
mean concave points     0.104
worst radius            0.076
dtype: float64


# Part B - Regression

In [10]:
data = load_diabetes(as_frame=True)
X = data.data
y = data.target        # continuous: a disease-progression score
print("X shape ->", X.shape, "| target range ->",
      f"{y.min():.0f} to {y.max():.0f}")
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE)   # no stratify for regression

X shape -> (442, 10) | target range -> 25 to 346


In [11]:
# Baseline linear model
lin = LinearRegression()
lin.fit(X_train, y_train)
pred_lin = lin.predict(X_test)
 
# Random Forest regressor
rfr = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)
rfr.fit(X_train, y_train)
pred_rfr = rfr.predict(X_test)
 
# Evaluate. For regression: RMSE, MAE, R².
def regression_scores(name, y_true, pred):
    mse = mean_squared_error(y_true, pred)
    rmse = np.sqrt(mse)     # portable RMSE (newer sklearn also has root_mean_squared_error)
    print(f"\n{name}")
    print(f"  RMSE : {rmse:.2f}   (same units as target; punishes big misses)")
    print(f"  MAE  : {mean_absolute_error(y_true, pred):.2f}   (robust to outliers)")
    print(f"  R2   : {r2_score(y_true, pred):.3f}   (1=perfect, 0=mean baseline)")
 
regression_scores("Linear Regression", y_test, pred_lin)
regression_scores("Random Forest Regressor", y_test, pred_rfr)
 
# Linear model coefficients — direction + magnitude of each feature's effect.
coefs = pd.Series(lin.coef_, index=X.columns)
print("\nLinear regression coefficients (sorted):\n",
      coefs.sort_values(ascending=False).round(1))
 


Linear Regression
  RMSE : 53.37   (same units as target; punishes big misses)
  MAE  : 41.55   (robust to outliers)
  R2   : 0.485   (1=perfect, 0=mean baseline)

Random Forest Regressor
  RMSE : 54.48   (same units as target; punishes big misses)
  MAE  : 43.34   (robust to outliers)
  R2   : 0.463   (1=perfect, 0=mean baseline)

Linear regression coefficients (sorted):
 s5     695.8
bmi    532.0
s2     508.3
bp     381.6
s4     269.5
s3     117.0
age     47.7
s6      26.3
sex   -242.0
s1    -918.5
dtype: float64


# PART C — PIPELINE + COLUMNTRANSFORMER (mixed data types)

In [12]:
# Real datasets mix numeric and categorical columns AND have missing values.
# A ColumnTransformer applies different preprocessing to different columns,
# and a Pipeline chains preprocessing + model into ONE object. Benefits:
#   - no data leakage (everything fits on train only, automatically)
#   - one .fit() / .predict() call, and it works cleanly with cross-validation.
 
# Build a small mixed dataset with a couple of NaNs.
rng = np.random.RandomState(RANDOM_STATE)
n = 400
mixed = pd.DataFrame({
    "age": rng.randint(18, 70, n).astype(float),
    "income": rng.normal(50000, 15000, n).round(0),
    "city": rng.choice(["London", "Paris", "Berlin", "Madrid"], n),
    "plan": rng.choice(["basic", "premium"], n, p=[0.6, 0.4]),
})
# inject some missing values
mixed.loc[rng.choice(n, 20, replace=False), "income"] = np.nan
mixed.loc[rng.choice(n, 15, replace=False), "city"] = np.nan
# target: churn, loosely dependent on plan + age so the model can learn something
churn_prob = 0.15 + 0.25 * (mixed["plan"] == "basic") + 0.004 * (mixed["age"] - 40)
y = (rng.rand(n) < churn_prob.clip(0.05, 0.9)).astype(int)
print("Mixed dataset head:\n", mixed.head())
print("\nmissing per column:\n", mixed.isna().sum())
 
numeric_features = ["age", "income"]
categorical_features = ["city", "plan"]
 
# Numeric branch: impute missing with median, then scale.
numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
# Categorical branch: impute missing with most frequent, then one-hot encode.
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
# Combine the two branches by column.
preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])
# Full pipeline: preprocessing + model as one estimator.
clf = Pipeline([
    ("prep", preprocess),
    ("model", RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE)),
])
 
Xtr, Xte, ytr, yte = train_test_split(
    mixed, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)
clf.fit(Xtr, ytr)                       # one call handles impute+scale+encode+fit
pred = clf.predict(Xte)
print("\nPipeline test accuracy ->", round(accuracy_score(yte, pred), 3))
print("Pipeline test f1       ->", round(f1_score(yte, pred), 3))
 

Mixed dataset head:
     age   income    city     plan
0  56.0  32784.0  Madrid    basic
1  69.0  70844.0  Madrid    basic
2  46.0  37171.0  Berlin    basic
3  32.0      NaN  London    basic
4  60.0  63243.0  Madrid  premium

missing per column:
 age        0
income    20
city      15
plan       0
dtype: int64

Pipeline test accuracy -> 0.69
Pipeline test f1       -> 0.367


# PART D — CROSS-VALIDATION & GRIDSEARCHCV

In [13]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(clf, mixed, y, cv=cv, scoring="f1")
print("5-fold F1 scores ->", np.round(scores, 3))
print(f"mean F1 -> {scores.mean():.3f}  (+/- {scores.std():.3f})")
 
# GridSearchCV: try combinations of hyperparameters, pick the best by CV score.
# Note the 'model__' prefix -> targets the 'model' step inside the pipeline.
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 5, 10],
}
grid = GridSearchCV(clf, param_grid, cv=cv, scoring="f1", n_jobs=-1)
grid.fit(mixed, y)
print("\nbest params ->", grid.best_params_)
print(f"best CV F1  -> {grid.best_score_:.3f}")

5-fold F1 scores -> [0.286 0.242 0.3   0.304 0.383]
mean F1 -> 0.303  (+/- 0.046)

best params -> {'model__max_depth': 10, 'model__n_estimators': 100}
best CV F1  -> 0.333
